In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session with Iceberg configurations
spark = SparkSession.builder \
  .appName("IcebergLocalDevelopment") \
  .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1') \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()

25/04/24 22:03:45 WARN Utils: Your hostname, Central resolves to a loopback address: 127.0.1.1; using 172.28.38.136 instead (on interface eth0)
25/04/24 22:03:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/taylor/APACHE-SPARK-COM-DELTA-LAKE-E-APACHE-ICEBERG/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/taylor/.ivy2/cache
The jars for the packages stored in: /home/taylor/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2dc41ba8-e253-4ea5-b16d-4d82cc93ac29;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1!iceberg-spark-runtime-3.5_2.12.jar (4457ms)
:: resolution report :: resolve 1489ms :: artifacts dl 4461ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicte

In [2]:
spark


In [7]:
# Criando tabela de funcionários
spark.sql("""
    CREATE TABLE local.funcionarios_iceberg (
        id_funcionario INT,
        nome STRING,
        cargo STRING,
        departamento STRING,
        salario DECIMAL(10,2),
        data_admissao DATE,
        ativo BOOLEAN
    ) USING iceberg
""")

DataFrame[]

In [12]:
spark.sql("""
    INSERT INTO local.funcionarios_iceberg 
    (id_funcionario, nome, cargo, departamento, salario, data_admissao, ativo)
    VALUES 
        (1, 'João Silva', 'Analista', 'TI', 5000.00, DATE '2023-01-15', true),
        (2, 'Maria Santos', 'Gerente', 'RH', 8000.00, DATE '2022-06-20', true),
        (3, 'Pedro Oliveira', 'Desenvolvedor', 'TI', 6000.00, DATE '2023-03-10', true),
        (4, 'Ana Costa', 'Assistente', 'Financeiro', 3500.00, DATE '2023-11-05', false)
""")

DataFrame[]

In [13]:

# Promovendo funcionário e ajustando salário
spark.sql("""
    UPDATE local.funcionarios_iceberg 
    SET cargo = 'Gerente de TI',
        salario = 9000.00
    WHERE id_funcionario = 1
""")


DataFrame[]

In [14]:
# Removendo funcionários inativos
spark.sql("""
    DELETE FROM local.funcionarios_iceberg 
    WHERE ativo = false
""")

DataFrame[]

In [15]:
# Consultando funcionários por departamento
spark.sql("""
    SELECT 
        departamento,
        COUNT(*) as total_funcionarios,
        AVG(salario) as media_salarial,
        MAX(salario) as maior_salario,
        MIN(salario) as menor_salario
    FROM local.funcionarios_iceberg
    GROUP BY departamento
    ORDER BY media_salarial DESC
""").show()

+------------+------------------+--------------+-------------+-------------+
|departamento|total_funcionarios|media_salarial|maior_salario|menor_salario|
+------------+------------------+--------------+-------------+-------------+
|          RH|                 1|   8000.000000|      8000.00|      8000.00|
|          TI|                 2|   7500.000000|      9000.00|      6000.00|
+------------+------------------+--------------+-------------+-------------+

